# Runge's Phenomenon and Chebyshev Nodes

Adding more interpolation nodes ought to improve the fit. It does not always. On equally spaced nodes the error for a perfectly smooth function can grow without bound as the degree goes up. The first part of this notebook reproduces that failure; the rest is about why it happens and what to do instead.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
# Enable interactive ipywidgets sliders. The try/except lets this same
# notebook run in plain Jupyter (outside Colab) without error.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## The error formula

For $f \in C^{n+1}[a,b]$, the interpolation error at $x$ is

$$ f(x) - p_n(x) = \frac{f^{(n+1)}(\xi)}{(n+1)!}\prod_{k=0}^{n}(x - x_k). $$

Two things on the right change as $n$ grows. One is the derivative term $f^{(n+1)}(\xi)/(n+1)!$, which depends on $f$. The other is the node polynomial $\omega(x)=\prod_k (x-x_k)$, which depends only on where the nodes sit. Equispaced nodes make $|\omega|$ large near the endpoints, and for Runge's function the derivative term does not shrink fast enough to make up for that. Chebyshev nodes cluster near the endpoints and keep $\max_x|\omega(x)|$ as small as it can be.

In [ ]:
# ---------------------------------------------------------------------------
# Equally spaced nodes
# ---------------------------------------------------------------------------
# A degree-n polynomial interpolant is pinned down by n+1 data points.
# Node placement matters; this is the obvious choice.

def equispaced_nodes(a, b, n):
    """Return n+1 equally spaced nodes on the interval [a, b].

    Divide [a, b] into n equal pieces. This choice is the source of
    Runge's phenomenon shown below.

    Parameters
    ----------
    a, b : float   endpoints of the interval
    n    : int     polynomial degree (so we get n+1 nodes)
    """
    return np.linspace(a, b, n + 1)

## Chebyshev nodes

The alternative placement. On $[-1, 1]$ the Chebyshev nodes (first kind) are

$$ x_k = \cos\!\left(\frac{(2k+1)\,\pi}{2(n+1)}\right), \qquad k = 0, \dots, n, $$

the $x$-projections of equally spaced points on a semicircle, so they cluster near the endpoints. That clustering is what keeps the endpoint oscillations of equispaced interpolation under control. For a general interval, rescale by the affine map taking $[-1,1]$ to $[a,b]$.

In [ ]:
# ---------------------------------------------------------------------------
# Chebyshev nodes
# ---------------------------------------------------------------------------
def chebyshev_nodes(a, b, n):
    """Return n+1 Chebyshev nodes (first kind) on [a, b].

    Parameters
    ----------
    a, b : float   endpoints of the interval
    n    : int     polynomial degree (so we get n+1 nodes)
    """
    k = np.arange(n + 1)
    x_std = np.cos((2 * k + 1) / (2 * (n + 1)) * np.pi)
    return 0.5 * (a + b) + 0.5 * (b - a) * x_std

## The Lagrange form

The interpolant through $(x_i, y_i)$ can be written down directly:

$$ p(x) = \sum_{i} y_i \, L_i(x), \qquad L_i(x) = \prod_{j \ne i} \frac{x - x_j}{x_i - x_j}, $$

where $L_i$ is $1$ at node $i$ and $0$ at every other node. Direct to express, though not the fastest; Newton's divided-difference form (next notebook) is cheaper to evaluate.

In [ ]:
# ---------------------------------------------------------------------------
# The interpolating polynomial, evaluated via the Lagrange formula
# ---------------------------------------------------------------------------
def lagrange_eval(nodes, values, xq):
    """Evaluate the polynomial interpolating (nodes[i], values[i]) at xq.

    Parameters
    ----------
    nodes  : array (m,)  distinct x-coordinates of the data
    values : array (m,)  y-coordinates, values[i] = f(nodes[i])
    xq     : array       points at which to evaluate the interpolant
    """
    nodes  = np.asarray(nodes,  dtype=float)
    values = np.asarray(values, dtype=float)
    xq     = np.asarray(xq,     dtype=float)

    result = np.zeros_like(xq)
    m = len(nodes)
    for i in range(m):
        Li = np.ones_like(xq)
        for j in range(m):
            if j != i:
                Li = Li * (xq - nodes[j]) / (nodes[i] - nodes[j])
        result = result + values[i] * Li
    return result

In [ ]:
# ---------------------------------------------------------------------------
# Test functions to interpolate
# ---------------------------------------------------------------------------
def runge(x):
    """Runge's function 1/(1 + 25 x^2) on [-1, 1].

    Smooth on the whole interval, yet equispaced interpolation of it diverges
    as the degree grows.
    """
    return 1.0 / (1.0 + 25.0 * x**2)

def sine(x):
    """A function that equispaced interpolation handles without trouble, for contrast."""
    return np.sin(np.pi * x)

# Registry so the interactive dropdown can pick a function by name.
TEST_FUNCTIONS = {"Runge 1/(1+25x^2)": runge, "sin(pi x)": sine}
NODE_BUILDERS  = {"equispaced": equispaced_nodes, "Chebyshev": chebyshev_nodes}

In [ ]:
# ---------------------------------------------------------------------------
# One figure: the function, its interpolant, the nodes, and the error curve
# ---------------------------------------------------------------------------
def show_interpolation(degree=12, node_type="equispaced",
                       func_name="Runge 1/(1+25x^2)", a=-1.0, b=1.0):
    """Plot f, its degree-`degree` interpolant on the chosen nodes, and |f - p|.

    Also prints the maximum error over a fine grid, so you can watch the number
    grow (equispaced) or shrink (Chebyshev) as you slide the degree.
    """
    f       = TEST_FUNCTIONS[func_name]
    nodefn  = NODE_BUILDERS[node_type]

    nodes   = nodefn(a, b, degree)           # n+1 interpolation nodes
    values  = f(nodes)                       # sample the function there
    xx      = np.linspace(a, b, 1000)        # dense grid for plotting
    p       = lagrange_eval(nodes, values, xx)
    err     = np.abs(f(xx) - p)
    max_err = err.max()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    ax1.plot(xx, f(xx), "k-", lw=2, label="f(x)")
    ax1.plot(xx, p, "r--", lw=2, label=f"interpolant p (deg {degree})")
    ax1.plot(nodes, values, "bo", ms=6, label="nodes")
    ax1.set_title(f"{func_name}, {node_type} nodes")
    ax1.legend(loc="upper right"); ax1.set_xlabel("x")

    ax2.semilogy(xx, err + 1e-18, "m-")      # small offset avoids log of zero
    ax2.set_title(f"pointwise error |f - p|   (max = {max_err:.3e})")
    ax2.set_xlabel("x"); ax2.set_ylabel("error (log scale)")

    plt.tight_layout(); plt.show()

# A first static call so the notebook shows something before you touch sliders.
show_interpolation(degree=12, node_type="equispaced")

## The node polynomial

Everything that depends on node placement sits in $\omega(x)=\prod_k(x-x_k)$, so plot it. The cell below shows $|\omega|$ for both node families at the same degree and tabulates $\max_x|\omega(x)|$. For $n+1$ Chebyshev nodes on $[-1,1]$ that maximum is exactly $2^{-n}$, and no other choice of $n+1$ nodes beats it.

In [ ]:
def omega(nodes, x):
    """The node polynomial prod_k (x - x_k), evaluated on the grid x."""
    out = np.ones_like(x)
    for xk in nodes:
        out = out * (x - xk)
    return out

xx = np.linspace(-1, 1, 2000)
print(f"{'n':>3} {'equispaced':>13} {'Chebyshev':>13} {'2^-n':>13} {'ratio':>9}")
for n in [6, 10, 14, 18, 22]:
    we = np.abs(omega(equispaced_nodes(-1, 1, n), xx)).max()
    wc = np.abs(omega(chebyshev_nodes(-1, 1, n), xx)).max()
    print(f"{n:3d} {we:13.3e} {wc:13.3e} {2.0**(-n):13.3e} {we / wc:9.1f}")

n = 14
plt.figure()
plt.semilogy(xx, np.abs(omega(equispaced_nodes(-1, 1, n), xx)), "r-", label="equispaced")
plt.semilogy(xx, np.abs(omega(chebyshev_nodes(-1, 1, n), xx)), "b-", label="Chebyshev")
plt.xlabel("x"); plt.ylabel("|omega(x)|  (log scale)")
plt.title(f"node polynomial, n = {n}"); plt.legend(); plt.show()

## The bound is too pessimistic

Taking worst cases over $x$ and over $\xi$ separately gives

$$ \max_x |f - p_n| \leq \frac{1}{(n+1)!}\max_\xi |f^{(n+1)}(\xi)| \cdot \max_x|\omega(x)|. $$

For Runge's function $\max_{[-1,1]}|f^{(n+1)}|$ grows like $(n+1)!\,5^{\,n+1}$, since $f$ has poles at $\pm i/5$. Put that against $\max|\omega| = 2^{-n}$ and the bound behaves like $5\,(5/2)^{\,n}$, so it diverges for Chebyshev nodes as well, even though the measured Chebyshev error below converges geometrically. A divergent bound tells you nothing about the error it bounds. To see Runge's phenomenon you have to measure it.

## Error against degree

One degree at a time tells you very little. The next cell sweeps the degree and records $E_n = \max_x|f(x)-p_n(x)|$ on a fine grid, for both node families and both test functions.

In [ ]:
grid = np.linspace(-1, 1, 4001)

def max_error(f, nodefn, n):
    """Largest |f - p_n| over the fine grid, for the given node family."""
    nodes = nodefn(-1.0, 1.0, n)
    return np.abs(f(grid) - lagrange_eval(nodes, f(nodes), grid)).max()

degrees = list(range(4, 41, 2))
curves = {(fname, nname): [max_error(f, nodefn, n) for n in degrees]
          for fname, f in [("Runge", runge), ("sin", sine)]
          for nname, nodefn in [("equi", equispaced_nodes), ("Cheb", chebyshev_nodes)]}

print(f"{'n':>4} | {'Runge equi':>12} {'Runge Cheb':>12} | {'sin equi':>12} {'sin Cheb':>12}")
print("-" * 62)
for i, n in enumerate(degrees):
    if n % 4 == 0:
        print(f"{n:4d} | {curves[('Runge','equi')][i]:12.3e} {curves[('Runge','Cheb')][i]:12.3e}"
              f" | {curves[('sin','equi')][i]:12.3e} {curves[('sin','Cheb')][i]:12.3e}")

plt.figure()
styles = {("Runge","equi"): "r.-", ("Runge","Cheb"): "b.-",
          ("sin","equi"): "r.--", ("sin","Cheb"): "b.--"}
for key, vals in curves.items():
    plt.semilogy(degrees, vals, styles[key], label=f"{key[0]}, {key[1]}")
plt.xlabel("degree n"); plt.ylabel("max error (log scale)")
plt.title("interpolation error against degree"); plt.legend(); plt.show()

Four curves come out of the sweep. Runge with equispaced nodes rises geometrically, about a factor of $1.5$ per degree. Runge with Chebyshev nodes falls geometrically, about a factor of $0.82$ per degree, which is the rate the pole positions predict. For $\sin(\pi x)$ both families converge quickly, but the equispaced curve bottoms out near $10^{-12}$ around $n = 20$ and then climbs back to roughly $10^{-7}$ by $n = 40$. That climb is roundoff, not approximation error. Evaluating the Lagrange form on nearly coincident node differences throws away digits, and `float64` has only about 16 of them to spend. The Chebyshev curve reaches the same floor and stays on it.

## Controls

Take equispaced nodes on Runge's function and push the degree past about 15. The interpolant develops large oscillations near $x=\pm 1$ and the error curve turns upward. Switch to Chebyshev nodes at that same degree and the oscillations are gone, with the error several orders of magnitude smaller. On $\sin(\pi x)$ equispaced nodes are already fine.

In [ ]:
# ---------------------------------------------------------------------------
# Interactive version, drag the sliders in Colab
# ---------------------------------------------------------------------------
# Slide `degree` up with equispaced nodes on the Runge function and watch the
# error grow near the endpoints. Switch to Chebyshev and watch it collapse.
interact(
    show_interpolation,
    degree=IntSlider(min=2, max=40, step=1, value=12, description="degree n"),
    node_type=Dropdown(options=list(NODE_BUILDERS), value="equispaced",
                       description="nodes"),
    func_name=Dropdown(options=list(TEST_FUNCTIONS),
                        value="Runge 1/(1+25x^2)", description="function"),
    a=(-2.0, 0.0, 0.5), b=(0.0, 2.0, 0.5),
);

## Summary

Higher degree is not automatically better: on equispaced nodes, polynomial interpolation of a smooth function can diverge. The error formula shows why, since it isolates $\omega(x)$ as the one factor you control, and Chebyshev nodes are the placement that makes $\max_x|\omega|$ smallest. Fast convergence comes back as soon as you use them.

The sweep also turns up something unrelated to approximation: the Lagrange form has a roundoff floor of its own, visible once the error drops below about $10^{-12}$. Node placement returns as a design choice when we get to Gaussian quadrature.

## Things to try

- Raise the degree with equispaced nodes until the oscillations near $x=\pm 1$ first become visible. Switch to Chebyshev at that same degree and watch the right panel drop.
- Widen the interval with the $a$ and $b$ sliders. How much does the error grow at a fixed degree?
- Add a function of your own to `TEST_FUNCTIONS` and find out whether equispaced nodes converge for it.